In [1]:
# Importation des modules
# Import bibliothèque de manipulation de dataframe
import pandas as pd

# Import des bibliothèques de viz
import matplotlib.pyplot as plt
import seaborn as sns

# Import split data
from sklearn.model_selection import train_test_split

# Import modèles de ML Supervisé Régression
from sklearn.linear_model import LinearRegression

# Import modèles de ML Supervisé Classification
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

# Import modèle de ML NON Supervisé
from sklearn.neighbors import NearestNeighbors

# Import des métriques
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# Import outil standardisation de la donnée
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, FunctionTransformer

# Import pipeline
from sklearn.pipeline import Pipeline

from sklearn.base import BaseEstimator, TransformerMixin

# Gestion des warnings
import warnings

import ast

In [2]:
# Custom transformer for MultiLabelBinarizer
class MultiLabelBinarizerPipelineFriendly(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.mlb = MultiLabelBinarizer()

    def fit(self, X, y=None):
        self.mlb.fit(X)
        return self

    def transform(self, X):
        return self.mlb.transform(X)

    def get_feature_names_out(self, input_features=None):
        return self.mlb.classes_

In [3]:
# Récuperation du df
df = pd.read_csv('../ressources/df_v3.csv', sep=';', encoding='utf-8')
df.isna().sum()

tconst             0
frenchTitle        0
genres             0
averageRating      0
numVotes           0
actor1            25
actor2            73
actor3           110
decade             0
dtype: int64

In [4]:
# On remplace les valeurs manquantes par la valeur 'Unknwown'
df.dropna(subset=['actor2', 'actor3'], inplace=True)
print(df.isna().sum())
df.fillna('Unknown', inplace=True)

tconst           0
frenchTitle      0
genres           0
averageRating    0
numVotes         0
actor1           0
actor2           0
actor3           0
decade           0
dtype: int64


In [5]:
# On concatene les acteurs pour en faire une colonne unique pour la passer dans un MultiLabelBinarizer
df['actors'] = df['actor1'].map(lambda x: [x]) + df['actor2'].map(lambda x: [x]) + df['actor3'].map(lambda x: [x])

# On supprime les colonnes inutiles
df = df.drop(columns=['actor1', 'actor2', 'actor3'])

# on enleve les [] et les '' de la colonne actors
df['actors'] = df['actors'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

df

,tconst,frenchTitle,genres,averageRating,numVotes,decade,actors
0,tt0003643,La conscience vengeresse,"Drama, Horror, Crime",6.4,1531,1910,"[Henry B. Walthall, Spottiswoode Aitken, Blanc..."
1,tt0003772,Cinderella,"Drama, Fantasy",6.0,1116,1910,"[Mary Pickford, Owen Moore, Isabel Vernon]"
2,tt0004181,Judith de Béthulie,"Drama, History, War",6.2,1501,1910,"[Blanche Sweet, Henry B. Walthall, Mae Marsh]"
3,tt0004465,Les exploits d'Elaine,"Drama, Adventure, Action",6.3,1107,1910,"[Pearl White, Crane Wilbur, Paul Panzer]"
4,tt0004707,La Folle Aventure de Charlot et de Lolotte,"Comedy, Drama, Romance",6.2,3824,1910,"[Charles Chaplin, Marie Dressler, Mabel Normand]"
...,...,...,...,...,...,...,...
41972,tt9904802,Enemy Lines,"Drama, Action, War",4.6,2045,2020,"[Ed Westwick, John Hannah, Tom Wisdom]"
41973,tt9907782,Eight for Silver,"Mystery, Horror, Fantasy",6.2,20398,2020,"[Boyd Holbrook, Kelly Reilly, Alistair Petrie]"
41974,tt9908390,Le lion,Comedy,5.5,1515,2020,"[Dany Boon, Philippe Katerine, Anne Serra]"
41975,tt9916190,Safeguard,"Thriller, Adventure, Action, Crime",3.6,263,2020,"[Patrick Gallagher, Akie Kotabe, Sean Cronin]"


----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Preprocessor
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [6]:
X = df.drop(columns=['frenchTitle'])
films_non_standardise = X.iloc[:3]

In [7]:
# Fonction pour alourdir la valeur des colonnes
def multiply_block(X, factor):
    return X * factor

In [15]:
# Preprocessor pour standardiser les colonnes numériques
preprocessor = ColumnTransformer(
    transformers=[
        ('actors', Pipeline([
            ('mlb', MultiLabelBinarizerPipelineFriendly()),
            ('weight', FunctionTransformer(lambda x: multiply_block(x, 2))),
            ]), 'actors'),
        ('genres', MultiLabelBinarizerPipelineFriendly(), 'genres'),
        ('decade', Pipeline([
            ('encoder', OrdinalEncoder()),
            ('weight', FunctionTransformer(lambda x: multiply_block(x, 0.5))),
            ]), ['decade']),
        ('scaler', StandardScaler(), ['averageRating']),
        ('numVotes', Pipeline([
            ('scaler', StandardScaler()),
            ('weight', FunctionTransformer(lambda x: multiply_block(x, 0.2))),
            ]), ['numVotes'])
    ]
)


----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Pipeline
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [16]:
# Création du pipeline
pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('knn', NearestNeighbors(n_neighbors=11))
    ]
)

pipeline.fit(X)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('actors',
                                                  Pipeline(steps=[('mlb',
                                                                   MultiLabelBinarizerPipelineFriendly()),
                                                                  ('weight',
                                                                   FunctionTransformer(func=<function <lambda> at 0x0000024EECA2FE20>))]),
                                                  'actors'),
                                                 ('genres',
                                                  MultiLabelBinarizerPipelineFriendly(),
                                                  'genres'),
                                                 ('decade',
                                                  Pipeline(steps=[('encoder',
                                                                   OrdinalEncoder()),
                                                                  ('weight',
                                                                   FunctionTransformer(func=<function <lambda> at 0x0000024EECA2EDE0>))]),
                                                  ['decade']),
                                                 ('scaler', StandardScaler(),
                                                  ['averageRating']),
                                                 ('numVotes',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler()),
                                                                  ('weight',
                                                                   FunctionTransformer(func=<function <lambda> at 0x0000024EECA2CA40>))]),
                                                  ['numVotes'])])),
                ('knn', NearestNeighbors(n_neighbors=11))])

In [ ]:
point_test = X.iloc[:3]
# Prédiction des voisins les plus proches
X_test_transformed = pipeline.named_steps['preprocessor'].transform(point_test)

distances, indices = pipeline.named_steps['knn'].kneighbors(X_test_transformed)
# Affichage des indices des voisins les plus proches
print("Indices des voisins les plus proches :", indices)
# Affichage des distances des voisins les plus proches
print("Distances des voisins les plus proches :", distances)


Indices des voisins les plus proches : [[   0  125  229  379  964]
 [   1  794 1628  835  131]
 [   2    8 1701  818  312]]
Distances des voisins les plus proches : [[0.         4.7        4.82182538 4.89897949 5.22015325]
 [0.         3.78945906 4.58257569 4.69467784 4.70744092]
 [0.         4.51220567 4.6        5.0039984  5.01597448]]


In [17]:
titres = ['Spider-Man', "Intouchables", 'Avatar']  # ou d’autres

for titre in titres:
    film_cible = df[df['frenchTitle'].str.contains(titre, case=False, na=False)]
    if film_cible.empty:
        print(f"Film '{titre}' non trouvé.")
        continue

    idx_film = film_cible.index[0]
    film_non_standardise = df.drop(columns=['frenchTitle']).loc[[idx_film]]
    film_transforme = pipeline.named_steps['preprocessor'].transform(film_non_standardise)
    distances, indices = pipeline.named_steps['knn'].kneighbors(film_transforme)

    print(f"\n🎬 Film : {df.loc[idx_film, 'frenchTitle']} (Index: {idx_film})")
    print(f"  Note moyenne : {df.loc[idx_film, 'averageRating']}")
    neighbor_original_indices = X.iloc[indices[0]].index
    neighbor_info = df.loc[neighbor_original_indices][['frenchTitle', 'averageRating', 'numVotes', 'decade', 'genres', 'actors']]
    print("  Voisins :")
    display(neighbor_info)


🎬 Film : Spider-Man (Index: 11170)
  Note moyenne : 7.4
  Voisins :


,frenchTitle,averageRating,numVotes,decade,genres,actors
11170,Spider-Man,7.4,924798,2000,"Sci-Fi, Adventure, Action, Fantasy","[Tobey Maguire, Kirsten Dunst, Willem Dafoe]"
14373,Spider-Man 2,7.5,748023,2000,"Sci-Fi, Adventure, Action, Fantasy","[Tobey Maguire, Kirsten Dunst, Alfred Molina]"
16101,Spider-Man 3,6.3,669840,2000,"Sci-Fi, Adventure, Action, Fantasy","[Tobey Maguire, Kirsten Dunst, Topher Grace]"
24709,Aquaman,6.8,538359,2010,"Adventure, Action, Fantasy","[Jason Momoa, Amber Heard, Willem Dafoe]"
15898,John Carter,6.6,293272,2010,"Sci-Fi, Adventure, Action, ScienceFiction","[Taylor Kitsch, Lynn Collins, Willem Dafoe]"
28561,La Grande Muraille,5.9,153692,2010,"Adventure, Action, Fantasy","[Matt Damon, Tian Jing, Willem Dafoe]"
31435,Midnight Special,6.6,85579,2010,"Adventure, ScienceFiction, Drama, Sci-Fi, Mystery","[Michael Shannon, Joel Edgerton, Kirsten Dunst]"
10691,Small Soldiers,6.3,108706,1990,"Adventure, ScienceFiction, Action, Fantasy, Co...","[Kirsten Dunst, Gregory Smith, David Cross]"
16531,Daybreakers,6.4,138234,2000,"ScienceFiction, Action, Fantasy, Sci-Fi, Horror","[Ethan Hawke, Willem Dafoe, Sam Neill]"
8995,Jumanji,7.1,396462,1990,"Comedy, Adventure, Family, Fantasy","[Robin Williams, Kirsten Dunst, Bonnie Hunt]"



🎬 Film : Intouchables (Index: 26460)
  Note moyenne : 8.5
  Voisins :


,frenchTitle,averageRating,numVotes,decade,genres,actors
26460,Intouchables,8.5,979906,2010,"Comedy, Drama","[François Cluzet, Omar Sy, Anne Le Ny]"
36438,Demain tout commence,7.3,29519,2010,"Comedy, Drama","[Omar Sy, Clémence Poésy, Antoine Bertrand]"
24328,Les Petits Mouchoirs,7.1,27726,2010,"Comedy, Drama","[François Cluzet, Marion Cotillard, Benoît Mag..."
36441,Médecin de campagne,6.5,3421,2010,"Comedy, Drama","[François Cluzet, Marianne Denicourt, Christop..."
29188,Un métier sérieux,6.6,1939,2020,"Comedy, Drama","[Vincent Lacoste, François Cluzet, Louise Bour..."
8748,Les apprentis,6.9,1230,1990,"Comedy, Drama","[Marie Trintignant, François Cluzet, Guillaume..."
40354,Racine,6.4,1361,2010,"Comedy, Drama","[Omar Sy, Lionel Louis Basse, Fatoumata Diawara]"
11759,La Comédie de Terracina,6.8,228,1990,"Comedy, Drama","[François Cluzet, Isabella Ferrari, Margherita..."
24199,French Dream,6.3,1379,2020,"Comedy, Drama","[Audrey Lamy, François Cluzet, Chantal Neuwirth]"
8702,Le Vent du Wyoming,6.5,206,1990,"Comedy, Drama","[François Cluzet, Sarah-Jeanne Salvy, France C..."



🎬 Film : Avatar (Index: 17836)
  Note moyenne : 7.9
  Voisins :


,frenchTitle,averageRating,numVotes,decade,genres,actors
17836,Avatar,7.9,1432326,2000,"Adventure, Action, ScienceFiction, Fantasy","[Sam Worthington, Zoe Saldaña, Sigourney Weaver]"
26090,Avatar : La Voie de l'eau,7.5,535310,2020,"Adventure, Action, ScienceFiction, Fantasy","[Sam Worthington, Zoe Saldaña, Sigourney Weaver]"
34349,Les Gardiens de la Galaxie Vol. 2,7.6,796264,2010,"Comedy, Adventure, Action, ScienceFiction","[Chris Pratt, Zoe Saldaña, Dave Bautista]"
24003,Star Trek Into Darkness,7.7,504266,2010,"Sci-Fi, Adventure, Action, ScienceFiction","[Chris Pine, Zachary Quinto, Zoe Saldaña]"
18130,Le Choc des Titans,5.8,299423,2010,"Adventure, Action, Fantasy","[Sam Worthington, Liam Neeson, Ralph Fiennes]"
14523,Pirates des Caraïbes : La Malédiction du Black...,8.1,1268349,2000,"Adventure, Action, Fantasy","[Johnny Depp, Geoffrey Rush, Orlando Bloom]"
10669,"Star Wars, épisode III : La Revanche des Sith",7.6,886185,2000,"Adventure, Action, ScienceFiction, Fantasy","[Hayden Christensen, Natalie Portman, Ewan McG..."
5120,L'Empire contre-attaque,8.7,1444813,1980,"Adventure, Action, ScienceFiction, Fantasy","[Mark Hamill, Harrison Ford, Carrie Fisher]"
5492,Le Retour du Jedi,8.3,1167185,1980,"Adventure, Action, ScienceFiction, Fantasy","[Mark Hamill, Harrison Ford, Carrie Fisher]"
5790,"Aliens, le retour",8.4,809836,1980,"Adventure, ScienceFiction, Action, Thriller, H...","[Sigourney Weaver, Michael Biehn, Carrie Henn]"
